# Multimodal Analysis: Text and Image In-Depth

**Description:** This notebook demonstrates **Multimodal Analysis** using the **CLIP (Contrastive Language-Image Pre-Training)** model via the Hugging Face `transformers` library. Multimodal analysis involves processing and correlating data from different modalities, such as text and images. This is increasingly relevant in social science for analyzing social media content (memes, posts with images).

We will cover:
1.  **Zero-Shot Image Classification**: Classifying an image into categories without training a specific model for those categories.
2.  **Image-Text Matching**: Finding the best caption for an image from a list of options.
3.  **Synthetic Image Generation**: Creating simple images with text to test the model's capabilities.

# License
This notebook is licensed under the MIT License. See the LICENSE file in the repository root for details.

In [ ]:
# Setup
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q transformers torch pillow
    print('Packages installed.')

In [ ]:
import torch
from PIL import Image, ImageDraw, ImageFont
from transformers import pipeline
import matplotlib.pyplot as plt
import numpy as np

# Check device
device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'GPU' if device == 0 else 'CPU'}")

## 1. Load the Zero-Shot Image Classification Pipeline
We use the `zero-shot-image-classification` pipeline. By default, this uses a CLIP model (like `openai/clip-vit-base-patch32`). This model has been trained on 400 million (image, text) pairs and can map images and text to the same vector space.

In [ ]:
# Load pipeline
classifier = pipeline("zero-shot-image-classification", model="openai/clip-vit-base-patch32", device=device)

## 2. Create Synthetic Images
Since downloading images can be unreliable in some environments (e.g., restricted internet access), we will generate simple synthetic images using Python's `PIL` library. 

We will create two images:
1.  A "Protest" image (Text: "VOTE NOW", Red background)
2.  A "Nature" image (Green circle on Blue background)

In [ ]:
def create_protest_image():
    img = Image.new('RGB', (400, 400), color='white')
    d = ImageDraw.Draw(img)
    # Draw a sign
    d.rectangle([(50, 50), (350, 350)], fill='red')
    # Add text (simulating a protest sign)
    # Note: Default font is very small, we just draw simplistic text shapes or use default
    # For simplicity in this env, we rely on color/shape mostly, but let's try to write text
    d.text((150, 180), "VOTE", fill="white")
    d.text((150, 200), "NOW", fill="white")
    return img

def create_nature_image():
    img = Image.new('RGB', (400, 400), color='skyblue')
    d = ImageDraw.Draw(img)
    # Draw a sun/tree
    d.ellipse([(100, 100), (300, 300)], fill='green')
    return img

img_protest = create_protest_image()
img_nature = create_nature_image()

# Display
fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs[0].imshow(img_protest)
axs[0].set_title("Synthetic Image 1")
axs[0].axis('off')

axs[1].imshow(img_nature)
axs[1].set_title("Synthetic Image 2")
axs[1].axis('off')
plt.show()

## 3. Classify Image 1 (The "Protest" Sign)
We provide a list of candidate labels. The model calculates the similarity between the image and each label.

In [ ]:
labels = ["political protest", "beach vacation", "cooking class", "sports event"]
results = classifier(img_protest, candidate_labels=labels)

print("Image 1 Classification Results:")
for r in results:
    print(f"{r['label']}: {r['score']:.4f}")

## 4. Classify Image 2 (The "Nature" Scene)
Let's try different labels for the second image.

In [ ]:
labels_nature = ["urban city", "forest or nature", "indoor office", "space rocket"]
results_nature = classifier(img_nature, candidate_labels=labels_nature)

print("\nImage 2 Classification Results:")
for r in results_nature:
    print(f"{r['label']}: {r['score']:.4f}")

## 5. Practical Application: Filtering Social Media Images
Imagine you have a dataset of social media posts and you want to filter for posts related to "Environment". You can use this zero-shot capability to tag images automatically without needing a labeled training set.

In [ ]:
def filter_for_topic(image, topic_label, threshold=0.5):
    # We compare the topic against a generic negative class
    res = classifier(image, candidate_labels=[topic_label, "other"])
    # Find score for the topic
    score = next(item['score'] for item in res if item['label'] == topic_label)
    return score > threshold, score

is_env, score = filter_for_topic(img_nature, "environment")
print(f"Is Image 2 about environment? {is_env} (Score: {score:.3f})")

is_env_prot, score_prot = filter_for_topic(img_protest, "environment")
print(f"Is Image 1 about environment? {is_env_prot} (Score: {score_prot:.3f})")